# Binance USD-M Futures Classification Maps

This notebook shows how to get Binance official futures classification dictionaries from `exchangeInfo`:

- `underlyingType`: primary category, such as `COIN`, `EQUITY`, `PREMARKET`, `COMMODITY`, `INDEX`.
- `underlyingSubType`: official sector tags, such as `AI`, `Layer-1`, `Alpha`, `Pre-IPO`, `TradFi`.

The helper returns dictionaries where each key is a category/tag and each value is the sorted list of matching futures symbols.

## 1. Import Local Package

In [ ]:
import sys
from pathlib import Path

import pandas as pd


def find_repo_path():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/home/suncong/binance_klines_data_fetch"),
    ]
    for candidate in candidates:
        if (candidate / "binance_klines_data_fetch").is_dir():
            return candidate.resolve()
    raise RuntimeError("Could not find the binance_klines_data_fetch repo path")


repo_path = find_repo_path()
repo_path_str = str(repo_path)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)

from binance_klines_data_fetch import get_um_futures_classification_maps

print("Imported package from:", repo_path)

## 2. Fetch Classification Maps

This cell calls Binance USD-M Futures `/fapi/v1/exchangeInfo`. By default, it includes `TRADING` contracts with `contractType` equal to `PERPETUAL` or `TRADIFI_PERPETUAL`, so it covers crypto perpetuals plus TradFi and Pre-IPO perpetuals.

In [ ]:
classification_maps = get_um_futures_classification_maps(
    quote_assets=None,
    contract_types=("PERPETUAL", "TRADIFI_PERPETUAL"),
    unknown_label="UNKNOWN",
)

type_map = classification_maps["underlyingType"]
subtype_map = classification_maps["underlyingSubType"]

print("underlyingType groups:", len(type_map))
print("underlyingSubType groups:", len(subtype_map))

## 3. underlyingType: Primary Category To Symbols

In [ ]:
type_summary = pd.DataFrame(
    [
        {
            "underlyingType": label,
            "symbol_count": len(symbols),
            "sample_symbols": symbols[:20],
        }
        for label, symbols in type_map.items()
    ]
).sort_values(["symbol_count", "underlyingType"], ascending=[False, True])

display(type_summary.reset_index(drop=True))

In [ ]:
def symbols_frame(group_map, label):
    return pd.DataFrame({"symbol": group_map.get(label, [])})


for label in ["COIN", "EQUITY", "PREMARKET", "COMMODITY", "INDEX", "UNKNOWN"]:
    symbols = type_map.get(label, [])
    print(f"{label}: {len(symbols)} symbols")
    display(symbols_frame(type_map, label).head(50))

## 4. underlyingSubType: Official Sector Tag To Symbols

In [ ]:
subtype_summary = pd.DataFrame(
    [
        {
            "underlyingSubType": label,
            "symbol_count": len(symbols),
            "sample_symbols": symbols[:20],
        }
        for label, symbols in subtype_map.items()
    ]
).sort_values(["symbol_count", "underlyingSubType"], ascending=[False, True])

display(subtype_summary.reset_index(drop=True))

In [ ]:
for label in ["AI", "Layer-1", "Alpha", "Pre-IPO", "TradFi", "DeFi", "Meme", "UNKNOWN"]:
    symbols = subtype_map.get(label, [])
    print(f"{label}: {len(symbols)} symbols")
    display(symbols_frame(subtype_map, label).head(50))

## 5. Use The Dictionaries Directly

In [ ]:
ai_symbols = subtype_map.get("AI", [])
layer1_symbols = subtype_map.get("Layer-1", [])
pre_ipo_symbols = subtype_map.get("Pre-IPO", [])
coin_symbols = type_map.get("COIN", [])

print("AI symbols:", ai_symbols[:30])
print("Layer-1 symbols:", layer1_symbols[:30])
print("Pre-IPO symbols:", pre_ipo_symbols[:30])
print("COIN symbols:", coin_symbols[:30])

selected_symbols = sorted(set(ai_symbols) | set(layer1_symbols) | set(pre_ipo_symbols))
print("Combined selected symbols:", len(selected_symbols))
print(selected_symbols[:50])

## Optional Filters

Use these variants when you need a narrower universe:

```python
# USDT-quoted contracts only
maps = get_um_futures_classification_maps(quote_assets=["USDT"])

# Crypto perpetual contracts only, excluding TRADIFI_PERPETUAL
maps = get_um_futures_classification_maps(contract_types=("PERPETUAL",))
```